### The following code pertains to Fig. 3a, Fig. 3b, Fig. 3c, Fig. 3d, Fig. 3e, Fig. 3k, Supplementary Fig. 10e, Supplementary Fig. 12a, Supplementary Fig. 12b, and Supplementary Fig. 12c.

# Start

In [ ]:
library(GeneNMF)
library(Seurat)
library(ggplot2)
library(UCell)
library(patchwork)
library(Matrix)
library(RcppML)
library(viridis)
library(qs)
library(dplyr)
library(stringr)
library(circlize)
library(tidyr)
library(tibble)

In [ ]:
setwd('/mnt/data/khm_scRNA/huh7_zxs/result_zxs/')

In [ ]:
seurat_obj <- readRDS('/mnt/data/khm_scRNA/huh7_zxs/result_zxs/scdata_filter.rds')
seurat_obj

In [ ]:
seurat_obj$batch %>% unique()

In [ ]:
seurat_obj <- subset(seurat_obj,subset = batch %in% c('normal2','Tatin'))

In [ ]:
seurat_obj$batch %>% table()

In [ ]:
seu <- seurat_obj
ndim <- 15

In [ ]:
length(rownames(seurat_obj))

In [ ]:
seu <- FindVariableFeatures(seu, nfeatures = 3000) 
seu <- runNMF(seu, k = ndim, assay="RNA") 
seu@reductions$NMF

In [ ]:
seu <- RunUMAP(
    seu, 
    reduction = "NMF", 
    dims=1:ndim, 
    reduction.name = "NMF_UMAP", 
    reduction.key = "nmfUMAP_")

In [ ]:
library(dplyr, help, pos = 2, lib.loc = NULL)
seu@meta.data %>% head()

In [ ]:
options(repr.plot.width = 8,repr.plot.height = 8)
DimPlot(seu, reduction = "NMF_UMAP", 
        group.by = "batch", label=T) + 
  theme(aspect.ratio = 1,axis.text = element_blank(),
    axis.title = element_blank(),
    axis.ticks = element_blank()) +
  ggtitle("NMF UMAP")

### Supplementary 10e

In [ ]:
seu.list <- SplitObject(seu, split.by = "batch")
geneNMF.programs <- multiNMF(seu.list, 
                             assay="RNA", 
                             slot="data", 
                             k=5:10, 
                             nfeatures = 10000) 

geneNMF.metaprograms <- getMetaPrograms(
    geneNMF.programs,
    nMP=10,
    weight.explained = 0.7,
    max.genes=100)

In [ ]:
options(repr.plot.width = 16,repr.plot.height = 14)
ph <- plotMetaPrograms(
    geneNMF.metaprograms,
    show_rownames = FALSE,
    show_colnames = FALSE)

In [ ]:
geneNMF.metaprograms$metaprograms.metrics

In [ ]:
lapply(geneNMF.metaprograms$metaprograms.genes, head)

In [ ]:
geneNMF.metaprograms <- getMetaPrograms(
    geneNMF.programs,
    nMP=16,
    min.confidence = 0.8,
    weight.explained = 0.7,
    max.genes=100,
    remove.empty = TRUE)

In [ ]:
geneNMF.metaprograms$metaprograms.genes %>% unlist() %>% unique() %>% length

In [ ]:
geneNMF.metaprograms$metaprograms.metrics

In [ ]:
gradient_colors <- colorRampPalette(c("#3ab5b0", "#3d99be", "#56317a"))
colorRampPalette(colors = c("#5D8BBA","white","#BF5663"))(100)
viridis(100, option = "A", direction = -1)

In [ ]:
options(repr.plot.width = 16,repr.plot.height = 14)
ph <- plotMetaPrograms(
    mp.res = geneNMF.metaprograms,
    palette = c('white','#bfd3e6','#9ebcda','#8c96c6','#8c6bb1','#88419d','#810f7c','#4d004b')
)

In [ ]:
pdf(file = '../result_figs/NMF_heatmap_16mp.pdf',width = 12,height = 12)
ph
dev.off()

In [ ]:
library(UCell)
mp.genes <- geneNMF.metaprograms$metaprograms.genes
seu <- AddModuleScore_UCell(
    seu, 
    features = mp.genes, 
    assay="RNA", 
    ncores=20, 
    name = "")

In [ ]:
seu@meta.data %>% colnames()
seu@meta.data %>% head()

In [ ]:
matrix <- seu@meta.data[,names(mp.genes)]
#dimred <- scale(matrix)
dimred <- as.matrix(matrix)
colnames(dimred) <- paste0("MP_",seq(1, ncol(dimred)))
#New dim reduction
seu@reductions[["MPsignatures"]] <- new(
    "DimReduc",
    cell.embeddings = dimred,
    assay.used = "RNA",
    key = "MP_",
    global = FALSE)
set.seed(123)
seu <- RunUMAP(seu, reduction="MPsignatures", dims=1:length(seu@reductions[["MPsignatures"]]),
               metric = "euclidean", reduction.name = "umap_MP")

In [ ]:
meta_data %>% head()

In [ ]:
meta_data <- seu@meta.data
meta_data <- meta_data %>% 
    mutate(
        function_type = case_when(
            sgRNA_type %in% c("SREBF2", "HMGCR", "SQLE", "INSIG1") ~ 'chol_synthesis',
            sgRNA_type %in% c("LDLR", "NPC1L1", "NPC1") ~ 'chol_uptake',
            sgRNA_type %in% c("APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3") ~ 'chol_efflux',
            sgRNA_type %in% c("SOAT1") ~ 'chol_esterification',
            sgRNA_type %in% c("AAVS",'NT') ~ 'NT',
            TRUE ~ sgRNA_type
        ),
        sgRNA_type_tmp = ifelse(sgRNA_type %in% c("AAVS",'NT'),yes = 'NT',no = sgRNA_type)
    )
saveRDS(meta_data,'../result_figs/meta_data_GeneNMF.rds')
seu@meta.data <- meta_data
meta_data$function_type %>% table()
meta_data$sgRNA_type_tmp %>% table()
meta_data %>% head()

In [ ]:
library(viridis)
options(repr.plot.width = 24,repr.plot.height = 14)
FeaturePlot(seu, features = names(mp.genes), reduction = "umap_MP", ncol=4) &
  scale_color_viridis(option="B") &
  theme(aspect.ratio = 1, axis.text=element_blank(), axis.ticks=element_blank())
ggsave(filename = '../result_figs/NMF_umap_16mp.pdf',width = 12,height = 12)

In [ ]:
seu <- FindNeighbors(seu, reduction="MPsignatures", dims=1:length(seu@reductions[["MPsignatures"]]), verbose = FALSE)
seu <- FindClusters(seu, resolution = 0.7, verbose = FALSE)

In [ ]:
seu$seurat_clusters %>% table()

In [ ]:
seu@meta.data %>% head()

In [ ]:
options(repr.plot.width = 36,repr.plot.height = 10)
a <- DimPlot(seu, 
             reduction = "umap_MP", 
             group.by = "batch", 
             label=T) +
  theme(aspect.ratio = 1,
        axis.text = element_blank(),
        axis.title = element_blank(),
        axis.ticks = element_blank()) + 
  ggtitle("batch")

b <- DimPlot(seu, 
            reduction = "umap_MP", 
            group.by = "RNA_snn_res.0.7",label.size = 8, 
            label=T) + 
      theme(aspect.ratio = 1,
            axis.text = element_blank(),
            axis.title = element_blank(),
            axis.ticks = element_blank()) + 
      ggtitle("Cluster") + NoLegend()
c <- DimPlot(seu, 
            reduction = "umap_MP", 
            group.by = "sgRNA_type", 
            pt.size = 2,
            label=T) + 
      theme(aspect.ratio = 1,
            axis.text = element_blank(),
            axis.title = element_blank(),
            axis.ticks = element_blank()) + 
      ggtitle("sgRNA_type") + NoLegend()
d <- DimPlot(seu, 
      reduction = "umap_MP", 
      group.by = "function_type", 
      pt.size = 2,
      label=T) + 
      theme(aspect.ratio = 1,
            axis.text = element_blank(),
            axis.title = element_blank(),
            axis.ticks = element_blank()) + 
      ggtitle("function_type")
a | b | c | d

In [ ]:
library(msigdbr)
library(fgsea)

top_p <- lapply(geneNMF.metaprograms$metaprograms.genes, function(program) {
  runGSEA(program, universe=rownames(seu), category = "H")
})

head(top_p$MP4)
saveRDS(top_p,'./top_p_H_GeneNMF.rds')

In [ ]:
top_p_GOBP <- lapply(geneNMF.metaprograms$metaprograms.genes, function(program) {
  runGSEA(program, universe=rownames(seu), , category = "C5", subcategory = "BP")
})

head(top_p_GOBP$MP4)
saveRDS(top_p_GOBP,'./top_p_GOBP_GeneNMF.rds')

In [ ]:
dir()

In [ ]:
saveRDS(seu,'./scdata_GeneNMF.rds')
save.image('./scdata_GeneNMF.RData')

In [ ]:
seq_along(top_p_GOBP)

In [ ]:
data_p_GOBP <- do.call(rbind, lapply(seq_along(top_p_GOBP), function(i) {
  df <- top_p_GOBP[[i]]
  df$MP <- names(top_p_GOBP)[i]
  df$overlapGenes <- sapply(df$overlapGenes, function(x) paste(unlist(x), collapse = ","))
  return(df)
}))
data_p_GOBP %>% head()
write.csv(data_p_GOBP,'../result_figs/data_top_p_GOBP.csv')

####  fig3b  right

In [ ]:
dir()
getwd()

In [ ]:
load('./scdata_GeneNMF.RData')

In [ ]:
top_p_GOBP <- readRDS('./top_p_GOBP_GeneNMF.rds')
top_p_GOBP %>% names()

In [ ]:
go_list <- list(
  MP12 = c(
      "GOBP_STEROL_BIOSYNTHETIC_PROCESS","GOBP_STEROL_METABOLIC_PROCESS","GOBP_LIPID_METABOLIC_PROCESS","GOBP_CELLULAR_RESPONSE_TO_INSULIN_STIMULUS"
  ),
  MP16 = c(
      "GOBP_REGULATION_OF_TRANSFERASE_ACTIVITY","GOBP_LOCOMOTION","GOBP_CELL_MIGRATION","GOBP_REGULATION_OF_PROTEIN_PHOSPHORYLATION"
  ),
  MP1 = c(
      "GOBP_MITOTIC_CELL_CYCLE_PROCESS","GOBP_CHROMOSOME_SEGREGATION","GOBP_CELL_CYCLE","GOBP_CELLULAR_COMPONENT_DISASSEMBLY"
  ),
  MP7 = c(
      "GOBP_CELLULAR_RESPONSE_TO_STRESS","GOBP_DNA_METABOLIC_PROCESS","GOBP_DNA_REPAIR","GOBP_CELLULAR_RESPONSE_TO_DNA_DAMAGE_STIMULUS"
  )
)

In [ ]:
custom_palette <- colorRampPalette(c("#B9B4D5", "#8B83B9"))
custom_palette(3)

In [ ]:
lapply(names(go_list),function(MP_select){
    print(MP_select)
    data_plot <- top_p_GOBP[[MP_select]] %>% 
        filter(pathway %in% go_list[[MP_select]]) %>% 
        arrange(desc(padj)) %>% 
        mutate(pathway = factor(pathway,levels = pathway %>% unique()))
    min_value <- data_plot$padj %>% {-log10(.) * 10 } %>% min() %>% ceiling() %>% {./10}
    max_value <- data_plot$padj %>% {-log10(.) * 10 } %>% max() %>% floor() %>% {./10}
    options(repr.plot.height = 5,repr.plot.width = 8)
    p <- ggplot(data_plot, aes(x = -log10(padj), y = pathway, fill = -log10(padj))) +
      geom_bar(stat = 'identity',width = 0.5) +
      geom_text(
          aes(x = 0.01, y = pathway,label = str_replace(pathway, "^\\s+", "") %>% str_to_title()),
          size = 7,angle = 0,hjust = 0,vjust = -2,color = 'black') +
      # scale_fill_manual(values = paletteer::paletteer_c("grDevices::Inferno", 30) %>%.[5:20] %>%  rev()) +
      scale_fill_gradient2(
            name = '-Log10(p.adjust)',
            low = custom_palette(3)[1],mid = custom_palette(3)[2],high = custom_palette(3)[3],
            midpoint = (min_value + max_value)/2,
          breaks = c(min_value,(min_value + max_value)/2,max_value),
          labels = c(min_value,(min_value + max_value)/2,max_value)
        ) +
      guides(fill = guide_colorbar(title.position = "left", title.theme = element_text(angle = 90,vjust = 0.5))) +
      scale_x_continuous(expand = c(0.01,0)) +
      scale_y_discrete(expand = c(0,0.4)) +
      labs(y = 'Pathway',x = 'RichFactor') +
      theme_classic(base_size = 20) +
      coord_cartesian(clip = "off") +
      theme(
        plot.margin = unit(c(60, 0, 0, 0), "pt"),
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24,hjust = 0),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.4, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.text.y = element_blank(),
        axis.title.x = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )
    file_path <- paste('../result_figs/NMF_GO_',MP_select,'.pdf',sep = '')
    ggsave(filename = file_path,plot = p,width = 8,height = 5)
    return(p)
})

In [ ]:
load('./scdata_GeneNMF.RData')

In [ ]:
metadata  <- seu@meta.data
metadata %>% dim()
metadata %>% head()

In [ ]:
MEs <- metadata %>% 
    select(starts_with("MP"))
MEs %>% head()

In [ ]:
set.seed(123) 

n_perm <- 10000 

results_lm_perm <- lapply(colnames(MEs), function(module){
  df <- data.frame(
    GeneScore = MEs[, module],
    perturbation = metadata$sgRNA_type,
    batch = metadata$batch,
    nGene = metadata$nFeature_RNA
  )
  df <- df[!is.na(df$perturbation), ]
  
  model <- lm(GeneScore ~ perturbation + batch + nGene, data = df)
  coefs <- summary(model)$coefficients
  perturbation_effects <- coefs[grep("perturbation", rownames(coefs)), , drop = FALSE]
  
  pert_names <- rownames(perturbation_effects)
  
  orig_tvals <- abs(perturbation_effects[, "t value"])
  
  perm_tvals <- matrix(NA, nrow = length(pert_names), ncol = n_perm)
  rownames(perm_tvals) <- pert_names
  
  for (i in 1:n_perm) {
    df$perturbation_perm <- unlist(tapply(df$perturbation, df$batch, sample))  
    perm_model <- lm(GeneScore ~ perturbation_perm + batch + nGene, data = df)
    perm_coefs <- summary(perm_model)$coefficients
    perm_perturbation <- perm_coefs[grep("perturbation_perm", rownames(perm_coefs)), , drop = FALSE]
    row.names(perm_perturbation) <- row.names(perm_perturbation) %>% str_remove('_perm')
    match_idx <- match(pert_names, rownames(perm_perturbation))
    perm_tvals[, i] <- abs(perm_perturbation[match_idx, "t value"])
  }
  
  empirical_p <- sapply(1:length(pert_names), function(j){
    mean(c(perm_tvals[j, ], orig_tvals[j]) >= orig_tvals[j])
  })
  names(empirical_p) <- pert_names
  
  list(
    perturbation_effects = perturbation_effects,
    empirical_p = empirical_p
  )
})

In [ ]:
names(results_lm_perm)

In [ ]:
names(results_lm_perm) <- colnames(MEs)
names(results_lm_perm)

In [ ]:
results_lm_perm[[1]][2] %>% head()

In [ ]:
lm_results_df <- do.call(rbind, lapply(names(results_lm_perm), function(mod){
  eff <- results_lm_perm[[mod]]$perturbation_effects
  pval_empirical <- results_lm_perm[[mod]]$empirical_p
  data.frame(
    module = mod,
    term = rownames(eff),
    estimate = eff[, "Estimate"],
    t_value = eff[, "t value"],
    p_value = eff[, "Pr(>|t|)"],
    empirical_p = pval_empirical[rownames(eff)]
  )
}))

lm_results_df$fdr_empirical_p <- p.adjust(lm_results_df$empirical_p, method = "BH")
lm_results_df %>% head()

In [ ]:
write.csv(lm_results_df,'./lm_results_df.csv')

In [ ]:
library(lme4)
library(lmerTest)

results_lmm <- lapply(colnames(MEs), function(module){
  df <- data.frame(
    GeneScore = MEs[, module],
    perturbation = metadata$sgRNA_type,
    batch = metadata$batch,
    nGene = metadata$nFeature_RNA
  )
  df <- df[!is.na(df$perturbation), ]
  
  df$GeneScore <- scale(df$GeneScore)
  df$nGene <- scale(df$nGene)
  
  model <- lmer(GeneScore ~ perturbation + batch + nGene + (1 | batch:perturbation), data = df)
  coefs <- summary(model)$coefficients
  
  perturbation_effects <- coefs[grep("perturbation", rownames(coefs)), , drop = FALSE]
  
  perturbation_effects
})


In [ ]:
names(results_lmm) <- colnames(MEs)
names(results_lmm)
results_lmm[[1]] %>% head()

In [ ]:
library(lme4)
library(lmerTest)

set.seed(123) 

n_perm <- 10000

results_lmm_perm <- lapply(colnames(MEs), function(module){
  df <- data.frame(
    GeneScore = MEs[, module],
    perturbation = metadata$sgRNA_type,
    batch = metadata$batch,
    nGene = metadata$nFeature_RNA
  )
  df <- df[!is.na(df$perturbation), ]
  
  df$GeneScore <- scale(df$GeneScore)
  df$nGene <- scale(df$nGene)
  
  model <- lmer(GeneScore ~ perturbation + batch + nGene + (1 | batch:perturbation), data = df)
  coefs <- summary(model)$coefficients
  
  perturbation_effects <- coefs[grep("perturbation", rownames(coefs)), , drop = FALSE]
  
  pert_names <- rownames(perturbation_effects)
  
  orig_tvals <- abs(perturbation_effects[, "t value"])
  
  perm_tvals <- matrix(NA, nrow = length(pert_names), ncol = n_perm)
  rownames(perm_tvals) <- pert_names
  
  for (i in 1:n_perm) {
    df$perturbation_perm <- unlist(tapply(df$perturbation, df$batch, sample))
    
    perm_model <- lmer(GeneScore ~ perturbation_perm + batch + nGene + (1 | batch:perturbation_perm), data = df)
    perm_coefs <- summary(perm_model)$coefficients
    perm_perturbation <- perm_coefs[grep("perturbation_perm", rownames(perm_coefs)), , drop = FALSE]
    row.names(perm_perturbation) <- row.names(perm_perturbation) %>% str_remove('_perm')
    match_idx <- match(pert_names, rownames(perm_perturbation))
    perm_tvals[, i] <- abs(perm_perturbation[match_idx, "t value"])
  }
  
  empirical_p <- sapply(1:length(pert_names), function(j){
    mean(c(perm_tvals[j, ], orig_tvals[j]) >= orig_tvals[j])
  })
  names(empirical_p) <- pert_names
  
  list(
    perturbation_effects = perturbation_effects,
    empirical_p = empirical_p
  )
})



In [ ]:
lmm_results_df <- do.call(rbind, lapply(names(results_lmm_perm), function(mod){
  eff <- results_lmm_perm[[mod]]$perturbation_effects
  pval_empirical <- results_lmm_perm[[mod]]$empirical_p
  data.frame(
    module = mod,
    term = rownames(eff),
    estimate = eff[, "Estimate"],
    t_value = eff[, "t value"],
    p_value = eff[, "Pr(>|t|)"],
    empirical_p = pval_empirical[rownames(eff)]
  )
}))

lmm_results_df$fdr_empirical_p <- p.adjust(lmm_results_df$empirical_p, method = "BH")

In [ ]:
write.csv(lmm_results_df,'./lmm_results_df.csv')

In [ ]:
load('./scdata_GeneNMF.RData')

In [ ]:
metadata  <- seu@meta.data %>% 
    filter(batch == 'normal2')
metadata$batch %>% table()
metadata %>% dim()
metadata %>% head()
metadata$batch %>% table()

In [ ]:
MEs <- metadata %>% 
    select(starts_with("MP"))
MEs %>% head()

In [ ]:
set.seed(123)  
n_perm <- 1000  
results_lm_perm <- lapply(colnames(MEs), function(module){
  df <- data.frame(
    GeneScore = MEs[, module],
    perturbation = metadata$sgRNA_identity,
    batch = metadata$batch,
    nGene = metadata$nFeature_RNA
  ) %>% 
    mutate(
        perturbation = case_when(
            perturbation %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ perturbation
        ),
        perturbation = factor(perturbation),
        perturbation = relevel(perturbation, ref = "NT")
    ) %>% 
    filter(!is.na(perturbation))
  
  model <- lm(GeneScore ~ perturbation + nGene, data = df)
  coefs <- summary(model)$coefficients
  perturbation_effects <- coefs[grep("perturbation", rownames(coefs)), , drop = FALSE]
  
  pert_names <- rownames(perturbation_effects)
  
  orig_tvals <- abs(perturbation_effects[, "t value"])
  
  perm_tvals <- matrix(NA, nrow = length(pert_names), ncol = n_perm)
  rownames(perm_tvals) <- pert_names
  
  for (i in 1:n_perm) {
    df$perturbation_perm <- sample(df$perturbation)  # batch内置换
    perm_model <- lm(GeneScore ~ perturbation_perm + nGene, data = df)
    perm_coefs <- summary(perm_model)$coefficients
    perm_perturbation <- perm_coefs[grep("perturbation_perm", rownames(perm_coefs)), , drop = FALSE]
    row.names(perm_perturbation) <- row.names(perm_perturbation) %>% str_remove('_perm')
    match_idx <- match(pert_names, rownames(perm_perturbation))
    perm_tvals[, i] <- abs(perm_perturbation[match_idx, "t value"])
  }
  
  empirical_p <- sapply(1:length(pert_names), function(j){
    mean(c(perm_tvals[j, ], orig_tvals[j]) >= orig_tvals[j])
  })
  names(empirical_p) <- pert_names
  
  list(
    perturbation_effects = perturbation_effects,
    empirical_p = empirical_p
  )
})

In [ ]:
names(results_lm_perm)

In [ ]:
names(results_lm_perm) <- colnames(MEs)
names(results_lm_perm)

In [ ]:
results_lm_perm[[1]][2] %>% head()

In [ ]:
lm_results_df <- do.call(rbind, lapply(names(results_lm_perm), function(mod){
  eff <- results_lm_perm[[mod]]$perturbation_effects
  pval_empirical <- results_lm_perm[[mod]]$empirical_p
  data.frame(
    module = mod,
    term = rownames(eff),
    estimate = eff[, "Estimate"],
    t_value = eff[, "t value"],
    p_value = eff[, "Pr(>|t|)"],
    empirical_p = pval_empirical[rownames(eff)]
  )
}))

lm_results_df$fdr_empirical_p <- p.adjust(lm_results_df$empirical_p, method = "BH")
lm_results_df %>% head()

In [ ]:
write.csv(lm_results_df,'./lm_results_df_normal_sgRNA_identity.csv')

In [ ]:
lm_results_df <- read.csv('./lm_results_df_normal_sgRNA_identity.csv',row.names = 1)
lm_results_df %>% head()

In [ ]:
data_plot$term %>% unique() %>% sort()

In [ ]:
genes <- c("SREBF2", "HMGCR", "SQLE", "INSIG1",
           "LDLR", "NPC1L1", "NPC1",
           "APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3",
           "SOAT1")
sgIdentity_sort <- lapply(genes,function(x) paste(x,c('-sg1','-sg2','-sg3'),sep = '')) %>% unlist()

In [ ]:
library(stringr)
data_plot <- lm_results_df %>% 
    mutate(
        p_log = -log10(empirical_p),
        is_sig = empirical_p < 0.01,
        term = term %>% str_remove('perturbation'),
        module = factor(module,levels = paste('MP',1:16,sep = ''))
    ) %>% 
    group_by(module) %>%
    mutate(
        term = case_when(
            term %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ term
        ),
        term = factor(term,levels = sgIdentity_sort),
        estimate_norm = estimate
    ) %>%
    group_by(module) %>% 
    mutate(
        estimate_scale = estimate_norm#scale(estimate_norm)
    )
data_plot %>% head()

In [ ]:
data_plot$term %>% unique()

In [ ]:
colnames(data_plot)

In [ ]:
summary(data_plot$fdr_empirical_p)
summary(data_plot$estimate_norm)
summary(data_plot$estimate_scale)
data_plot %>% 
    filter(term == 'NT') %>% 
    pull(estimate_norm) %>% unique()

In [ ]:
data_plot <- data_plot %>% 
    filter(term != 'NT')

In [ ]:
library(ggplot2)
options(repr.plot.width = 32, repr.plot.height = 10)
ggplot(data_plot, aes(x = term, y = module)) +
  geom_point(aes(size = p_log, color = estimate_scale)) +
  geom_point(data = subset(data_plot, is_sig),
             aes(size = p_log, fill = estimate_scale), 
             shape = 21, color = "black", stroke = 2,
             show.legend = FALSE) +  
  scale_color_gradient2(low = "#1A5592",mid = "grey98", high = "#B83D3D") +
  scale_fill_gradient2(low = "#1A5592",mid = "grey98", high = "#B83D3D") +
  scale_size_continuous(range = c(7, 10)) +
  labs(x = "Perturbation", y = "Cell Type", color = "Effect Size", fill = "Effect Size", size = "-log10(P.adj)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),
    axis.title = element_blank(),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30, hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )
ggsave('../result_figs/lm_normal_sg1-3.pdf',width = 32,height = 10)

In [ ]:
data_plot %>% head()

In [ ]:
data_plot_sub <- data_plot %>% 
    filter(module %in% c('MP1','MP7','MP12','MP16')) %>% 
    mutate(
        module = factor(module,levels = c('MP12','MP16','MP1','MP7') %>% rev()),
        facet_group = ifelse(term %in% sgIdentity_sort[1:21],yes = 'Group1',no = 'Group2')
    )

In [ ]:
library(ggplot2)
options(repr.plot.width = 16, repr.plot.height = 8)
ggplot(data_plot_sub,aes(x = term, y = module)) +
  geom_point(aes(size = p_log, color = estimate_scale)) +#, fill = estimate_scale
  geom_point(data = subset(data_plot_sub, is_sig),
             aes(size = p_log, color = estimate_scale), 
             shape = 21, color = "black", stroke = 2,
             show.legend = FALSE) +
  scale_color_gradient2(
      low = "#1A5592",mid = "grey98", high = "#B83D3D",
      breaks = c(-0.01, 0.02, 0.05),
      labels = c("-0.01", "0.02", "0.05")
  ) +  
  scale_size_continuous(range = c(7, 10)) +
  # guides(fill = guide_colorbar(br))
  facet_wrap(~ facet_group,scales = 'free_x',ncol = 1) +
  labs(x = "Perturbation", y = "Cell Type", color = "Effect Size", fill = "Effect Size", size = "-log10(P-value)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),  # <- 改成linewidth
    axis.title = element_blank(),
    axis.line = element_blank(),
    # axis.ticks = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30, hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )
ggsave('../result_figs/lm_normal_sg1-3_4mp.pdf',width = 16,height = 8)

In [ ]:
metadata  <- seu@meta.data %>% 
    filter(batch == 'Tatin')
metadata$batch %>% table()
metadata %>% dim()
metadata %>% head()
metadata$batch %>% table()

In [ ]:
MEs <- metadata %>% 
    select(starts_with("MP"))
MEs %>% head()

In [ ]:
module <- colnames(MEs)[1]
df <- data.frame(
    GeneScore = MEs[, module],
    perturbation = metadata$sgRNA_identity,
    batch = metadata$batch,
    nGene = metadata$nFeature_RNA
) %>% 
mutate(
    perturbation = case_when(
        perturbation %in% c('NT1','NT2','AAVS') ~ 'NT',
        TRUE ~ perturbation
    ),
    perturbation = factor(perturbation),
    perturbation = relevel(perturbation, ref = "NT")
) %>% 
filter(!is.na(perturbation))

model <- lm(GeneScore ~ perturbation + nGene, data = df)
coefs <- summary(model)$coefficients
perturbation_effects <- coefs[grep("perturbation", rownames(coefs)), , drop = FALSE]

In [ ]:
perturbation_effects %>% head()

In [ ]:
set.seed(123) 

n_perm <- 10000  

results_lm_perm <- lapply(colnames(MEs), function(module){
  df <- data.frame(
    GeneScore = MEs[, module],
    perturbation = metadata$sgRNA_identity,
    batch = metadata$batch,
    nGene = metadata$nFeature_RNA
  ) %>% 
    mutate(
        perturbation = case_when(
            perturbation %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ perturbation
        ),
        perturbation = factor(perturbation),
        perturbation = relevel(perturbation, ref = "NT")
    ) %>% 
    filter(!is.na(perturbation))
  
  model <- lm(GeneScore ~ perturbation + nGene, data = df)
  coefs <- summary(model)$coefficients
  perturbation_effects <- coefs[grep("perturbation", rownames(coefs)), , drop = FALSE]
  
  pert_names <- rownames(perturbation_effects)
  
  orig_tvals <- abs(perturbation_effects[, "t value"])
  
  perm_tvals <- matrix(NA, nrow = length(pert_names), ncol = n_perm)
  rownames(perm_tvals) <- pert_names
  
  for (i in 1:n_perm) {
    df$perturbation_perm <- sample(df$perturbation) 
    perm_model <- lm(GeneScore ~ perturbation_perm + nGene, data = df)
    perm_coefs <- summary(perm_model)$coefficients
    perm_perturbation <- perm_coefs[grep("perturbation_perm", rownames(perm_coefs)), , drop = FALSE]
    row.names(perm_perturbation) <- row.names(perm_perturbation) %>% str_remove('_perm')
    match_idx <- match(pert_names, rownames(perm_perturbation))
    perm_tvals[, i] <- abs(perm_perturbation[match_idx, "t value"])
  }
  
  empirical_p <- sapply(1:length(pert_names), function(j){
    mean(c(perm_tvals[j, ], orig_tvals[j]) >= orig_tvals[j])
  })
  names(empirical_p) <- pert_names
  
  list(
    perturbation_effects = perturbation_effects,
    empirical_p = empirical_p
  )
})

In [ ]:
names(results_lm_perm)

In [ ]:
names(results_lm_perm) <- colnames(MEs)
names(results_lm_perm)

In [ ]:
results_lm_perm[[1]][2] %>% head()

In [ ]:
lm_results_df <- do.call(rbind, lapply(names(results_lm_perm), function(mod){
  eff <- results_lm_perm[[mod]]$perturbation_effects
  pval_empirical <- results_lm_perm[[mod]]$empirical_p
  data.frame(
    module = mod,
    term = rownames(eff),
    estimate = eff[, "Estimate"],
    t_value = eff[, "t value"],
    p_value = eff[, "Pr(>|t|)"],
    empirical_p = pval_empirical[rownames(eff)]
  )
}))

lm_results_df$fdr_empirical_p <- p.adjust(lm_results_df$empirical_p, method = "BH")
lm_results_df %>% head()

In [ ]:
write.csv(lm_results_df,'./lm_results_df_Tatin_sgRNA_identity_100000.csv')

In [ ]:
lm_results_df_normal <- read.csv('./lm_results_df_Normal_sgRNA_identity.csv',row.names = 1) %>% 
    mutate(group = 'normal') %>% rownames_to_column('rowname')
lm_results_df_tatin <- read.csv('./lm_results_df_Tatin_sgRNA_identity.csv',row.names = 1) %>% 
    mutate(group = 'tatin') %>% rownames_to_column('rowname')
lm_results_df <- rbind(lm_results_df_normal,lm_results_df_tatin)
lm_results_df %>% head()

In [ ]:
genes <- c("SREBF2", "HMGCR", "SQLE", "INSIG1",
           "LDLR", "NPC1L1", "NPC1",
           "APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3",
           "SOAT1")
sgIdentity_sort <- lapply(genes,function(x) paste(x,c('-sg1','-sg2','-sg3'),sep = '')) %>% unlist()

In [ ]:
p.adjust(c(0.05,0.01,0.1,1), method = "BH")

In [ ]:
lm_results_df %>% colnames()

In [ ]:
(names(lm_results_df))

In [ ]:
library(stringr)
data_plot <- lm_results_df %>% 
    mutate(
        p_log = -log10(empirical_p),
        is_sig = empirical_p < 0.01,
        term = term %>% str_remove('perturbation'),
        module = factor(module,levels = paste('MP',1:16,sep = ''))
    ) %>% 
    group_by(module) %>%
    mutate(
        term = case_when(
            term %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ term
        ),
        term = factor(term,levels = sgIdentity_sort),
        estimate_norm = estimate
    ) %>%
    group_by(module) %>%                               
      mutate(estimate_scale =                        
               estimate / max(abs(estimate), na.rm = TRUE)
             ) %>%
      ungroup()  %>% 
    filter(group == 'tatin') %>% 
    column_to_rownames('rowname')
data_plot %>% head()

In [ ]:
colnames(data_plot)

In [ ]:
summary(data_plot$fdr_empirical_p)
summary(data_plot$estimate_norm)
summary(data_plot$estimate_scale)
data_plot %>% 
    filter(term == 'NT') %>% 
    pull(estimate_norm) %>% unique()

In [ ]:
data_plot <- data_plot %>% 
    filter(term != 'NT')

### Supplementary Fig. 12 c

In [ ]:
library(ggplot2)
options(repr.plot.width = 32, repr.plot.height = 10)
ggplot(data_plot, aes(x = term, y = module)) +
  geom_point(aes(size = p_log, color = estimate_scale)) +
  geom_point(data = subset(data_plot, is_sig),
             aes(size = p_log, color = estimate_scale), 
             shape = 21, color = "black", stroke = 2,
             show.legend = FALSE) +
  scale_color_gradient2(low = "#1A5592",mid = "grey98", high = "#B83D3D") +
  scale_size_continuous(range = c(7, 10)) +
  labs(x = "Perturbation", y = "Cell Type", color = "Effect Size", fill = "Effect Size", size = "-log10(P-value)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),  # <- 改成linewidth
    axis.title = element_blank(),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30, hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )
ggsave('../result_figs/lm_tatin_sg1-3.pdf',width = 32,height = 10)

In [ ]:
data_plot_sub <- data_plot %>% 
    filter(module %in% c('MP1','MP7','MP12','MP16')) %>% 
    mutate(
        module = factor(module,levels = c('MP12','MP16','MP1','MP7') %>% rev()),
        facet_group = ifelse(term %in% sgIdentity_sort[1:21],yes = 'Group1',no = 'Group2')
    )
data_plot_sub %>% head()

In [ ]:
data_plot_sub <- data_plot_sub %>% 
    group_by(module) %>%                               
      mutate(estimate_scale =                        
               estimate / max(abs(estimate), na.rm = TRUE)
             ) %>%
      ungroup() 
data_plot_sub %>% head()

In [ ]:
data_plot_sub$estimate_scale %>% range()

### fig3e top

In [ ]:
library(ggplot2)
options(repr.plot.width = 16, repr.plot.height = 8)
ggplot(data_plot_sub,aes(x = term, y = module)) +
  geom_point(aes(size = p_log, color = estimate_scale)) +
  geom_point(data = subset(data_plot_sub, is_sig),
             aes(size = p_log, color = estimate_scale), 
             shape = 21, color = "black", stroke = 2,
             show.legend = FALSE) +
  scale_color_gradient2(
      low = "#1A5592",mid = "grey98", high = "#B83D3D",
      breaks = c(-0.8,0,1),
      labels = c(-0.8,0,1)
  ) +  
  scale_size_continuous(range = c(7, 10)) +
  facet_wrap(~ facet_group,scales = 'free_x',ncol = 1) +
  labs(x = "Perturbation", y = "Cell Type", color = "Effect Size", fill = "Effect Size", size = "-log10(P-value)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),  # <- 改成linewidth
    axis.title = element_blank(),
    axis.line = element_blank(),
    # axis.ticks = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30, hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )
ggsave('../result_figs/lm_tatin_sg1-3_4mp.pdf',width = 16,height = 8)

### Supplementary Fig. 12 b

In [ ]:
library(stringr)
data_plot <- lm_results_df %>% 
    mutate(
        p_log = -log10(empirical_p),
        is_sig = empirical_p < 0.01,
        term = term %>% str_remove('perturbation'),
        module = factor(module,levels = paste('MP',1:16,sep = ''))
    ) %>% 
    group_by(module) %>%
    mutate(
        term = case_when(
            term %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ term
        ),
        term = factor(term,levels = sgIdentity_sort),
        estimate_norm = estimate
    ) %>%
    group_by(module) %>%                               
      mutate(estimate_scale =                        
               estimate / max(abs(estimate), na.rm = TRUE)
             ) %>%
      ungroup()  %>% 
    filter(group == 'normal') %>% 
    column_to_rownames('rowname')
library(ggplot2)
options(repr.plot.width = 32, repr.plot.height = 10)
ggplot(data_plot, aes(x = term, y = module)) +
  geom_point(aes(size = p_log, color = estimate_scale)) +
  geom_point(data = subset(data_plot, is_sig),
             aes(size = p_log, color = estimate_scale), 
             shape = 21, color = "black", stroke = 2,
             show.legend = FALSE) +
  scale_color_gradient2(low = "#1A5592",mid = "grey98", high = "#B83D3D") +
  scale_size_continuous(range = c(7, 10)) +
  labs(x = "Perturbation", y = "Cell Type", color = "Effect Size", fill = "Effect Size", size = "-log10(P-value)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1), 
    axis.title = element_blank(),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30, hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )
ggsave('../result_figs/lm_normal_sg1-3.pdf',width = 32,height = 10)

In [ ]:
data_plot_sub <- data_plot %>% 
    filter(module %in% c('MP1','MP7','MP12','MP16')) %>% 
    mutate(
        module = factor(module,levels = c('MP12','MP16','MP1','MP7') %>% rev()),
        facet_group = ifelse(term %in% sgIdentity_sort[1:21],yes = 'Group1',no = 'Group2')
    )
data_plot_sub %>% head()

In [ ]:
data_plot_sub <- data_plot_sub %>% 
    group_by(module) %>%                               
      mutate(estimate_scale =                        
               estimate / max(abs(estimate), na.rm = TRUE)
             ) %>%
      ungroup() 
data_plot_sub %>% head()

In [ ]:
data_plot_sub$estimate_scale %>% range()

### fig3e bottom

In [ ]:
library(ggplot2)
options(repr.plot.width = 16, repr.plot.height = 8)
ggplot(data_plot_sub,aes(x = term, y = module)) +
  geom_point(aes(size = p_log, color = estimate_scale)) +
  geom_point(data = subset(data_plot_sub, is_sig),
             aes(size = p_log, color = estimate_scale), 
             shape = 21, color = "black", stroke = 2,
             show.legend = FALSE) +
  scale_color_gradient2(
      low = "#1A5592",mid = "grey98", high = "#B83D3D",
      breaks = c(-2,-1,0,1,2,3),
      labels = c(-2,-1,0,1,2,3)
  ) +  
  scale_size_continuous(range = c(7, 10)) +
  facet_wrap(~ facet_group,scales = 'free_x',ncol = 1) +
  labs(x = "Perturbation", y = "Cell Type", color = "Effect Size", fill = "Effect Size", size = "-log10(P-value)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),  # <- 改成linewidth
    axis.title = element_blank(),
    axis.line = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30, hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )
ggsave('../result_figs/lm_normal_sg1-3_4mp.pdf',width = 16,height = 8)

In [ ]:
DefaultAssay(seurat_obj) <- 'Protein'
seurat_obj <- NormalizeData(object = seurat_obj,normalization.method = "CLR", margin =2)
seurat_obj

In [ ]:
gene_vec <- c(
    "SREBF2-sg1","SREBF2-sg3","INSIG1-sg1","LDLR-sg1","LDLR-sg2","LDLR-sg3","NOC1L1-sg2",
    "NPC1-sg1","NPC1-sg2","NPC1-sg3","APOB-sg2","APOB-sg3","ABCA1-sg1","ABCA1-sg2","ABCG1-sg1",
    "ABCG1-sg2","ABCG1-sg3","ABCG5-sg2","ABCG5-sg3","ABCG8-sg2","SOAT1-sg1","SOAT1-sg2",
    "SOAT1-sg3","HMGCR-sg1","HMGCR-sg3","SQLE-sg3","MTTP-sg2",
    'NT1','NT2','AAVS'
)
seurat_use <- subset(seurat_obj,subset = sgRNA_identity %in% gene_vec)
seurat_use <- subset(seurat_use,subset = batch %in% c('Tatin'))

In [ ]:
seurat_use$batch %>% unique()

In [ ]:
feature <- rownames(seurat_use)
feature

In [ ]:
feature_use <- 'ALOD4-pAbO'

In [ ]:
seurat_use@meta.data %>% colnames()

In [ ]:
data_exp <- FetchData(
    object = seurat_use,
    vars = c(feature_use,'batch','sgRNA_type'),
    layer = "data"
) %>% 
    rownames_to_column('sample') %>% 
    pivot_longer(cols = all_of(feature_use)) %>% 
    dplyr::select(c('sample','value'))
data_exp %>% head()

In [ ]:
meta_data <- seurat_use@meta.data  %>% 
    rownames_to_column('sample') %>% 
    left_join(data_exp,by = 'sample') %>% 
    column_to_rownames('sample') %>% 
    arrange(desc(value)) %>% 
    mutate(
        ratio_alod4 = 1:n(),
        group_alod4_30 = case_when(
            ratio_alod4 <= (n()*0.2) ~ 'top 20%',
            ratio_alod4 > (n()*0.8) ~ 'bottom 20%',
            TRUE ~ 'middle'
        ),
        group_alod4_20 = case_when(
            ratio_alod4 <= (n()*0.2) ~ 'top 20%',
            ratio_alod4 > (n()*0.8) ~ 'bottom 20%',
            TRUE ~ 'middle'
        )
    ) 
seurat_use@meta.data <- meta_data
meta_data$group_alod4_20 %>% table()
meta_data$group_alod4_30 %>% table()
meta_data %>% head()

In [ ]:
metadata$group_alod4_30 %>% unique()

In [ ]:
metadata  <- seu@meta.data %>% 
    filter(batch == 'Tatin') %>% 
    rownames_to_column('Cellid') %>% 
    left_join(
        meta_data %>% 
            rownames_to_column('Cellid') %>% 
            select(Cellid,group_alod4_30),
        by = 'Cellid'
    ) %>% 
    mutate(
        sgRNA_identity = ifelse(sgRNA_identity%in% c('NT1','NT2','AAVS'),yes = 'NT',no = sgRNA_identity),
        group_alod4_30 = case_when(
            group_alod4_30 == 'top 20%' ~ 'top 20%',
            group_alod4_30 %in% c('middle','bottom 20%') ~ 'Use',
            TRUE ~ group_alod4_30
        ),
        sgRNA_identity = case_when(
            (!is.na(group_alod4_30) & (sgRNA_identity %in% c('SOAT1-sg1','SOAT1-sg2','SOAT1-sg3'))) ~ paste(sgRNA_identity,group_alod4_30,sep = '_'),
            TRUE ~ sgRNA_identity
        )
    )
metadata %>% dim()

In [ ]:
metadata$group_alod4_30 %>% unique()
metadata$sgRNA_identity %>% unique() %>% sort()

In [ ]:
metadata$sgRNA_identity %>% table()
metadata$group_alod4_30 %>% table()
MEs <- metadata %>% 
    select(starts_with("MP"))
MEs %>% dim()
MEs %>% head()

In [ ]:
set.seed(123)  

n_perm <- 10000  

results_lm_perm <- lapply(colnames(MEs), function(module){
  df <- data.frame(
    GeneScore = MEs[, module],
    perturbation = metadata$sgRNA_identity,
    batch = metadata$batch,
    nGene = metadata$nFeature_RNA
  ) %>% 
    mutate(
        perturbation = case_when(
            perturbation %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ perturbation
        ),
        perturbation = factor(perturbation),
        perturbation = relevel(perturbation, ref = "NT")
    ) %>% 
    filter(!is.na(perturbation))
  
  model <- lm(GeneScore ~ perturbation + nGene, data = df)
  coefs <- summary(model)$coefficients
  perturbation_effects <- coefs[grep("perturbation", rownames(coefs)), , drop = FALSE]
  
  pert_names <- rownames(perturbation_effects)
  
  orig_tvals <- abs(perturbation_effects[, "t value"])
  
  perm_tvals <- matrix(NA, nrow = length(pert_names), ncol = n_perm)
  rownames(perm_tvals) <- pert_names
  
  for (i in 1:n_perm) {
    df$perturbation_perm <- sample(df$perturbation)  # batch内置换
    perm_model <- lm(GeneScore ~ perturbation_perm + nGene, data = df)
    perm_coefs <- summary(perm_model)$coefficients
    perm_perturbation <- perm_coefs[grep("perturbation_perm", rownames(perm_coefs)), , drop = FALSE]
    row.names(perm_perturbation) <- row.names(perm_perturbation) %>% str_remove('_perm')
    match_idx <- match(pert_names, rownames(perm_perturbation))
    perm_tvals[, i] <- abs(perm_perturbation[match_idx, "t value"])
  }
  
  empirical_p <- sapply(1:length(pert_names), function(j){
    mean(c(perm_tvals[j, ], orig_tvals[j]) >= orig_tvals[j])
  })
  names(empirical_p) <- pert_names
  
  list(
    perturbation_effects = perturbation_effects,
    empirical_p = empirical_p
  )
})

In [ ]:
names(results_lm_perm) <- colnames(MEs)
names(results_lm_perm)

In [ ]:
lm_results_df <- do.call(rbind, lapply(names(results_lm_perm), function(mod){
  eff <- results_lm_perm[[mod]]$perturbation_effects
  pval_empirical <- results_lm_perm[[mod]]$empirical_p
  data.frame(
    module = mod,
    term = rownames(eff),
    estimate = eff[, "Estimate"],
    t_value = eff[, "t value"],
    p_value = eff[, "Pr(>|t|)"],
    empirical_p = pval_empirical[rownames(eff)]
  )
}))

lm_results_df$fdr_empirical_p <- p.adjust(lm_results_df$empirical_p, method = "BH")
lm_results_df %>% dim()
lm_results_df %>% head()

In [ ]:
write.csv(lm_results_df,'./lm_results_df_Tatin_sgRNA_identity_alod4—Group.csv')

In [ ]:
lm_results_df <- read.csv('./lm_results_df_Tatin_sgRNA_identity_alod4—Group.csv',row.names = 1)
lm_results_df %>% dim()
lm_results_df %>% head()

In [ ]:
lm_results_df$term %>% unique() %>% str_remove('perturbation')

In [ ]:
genes <- c("SREBF2", "HMGCR", "SQLE", "INSIG1",
           "LDLR", "NPC1L1", "NPC1",
           "APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3"
          )
sgIdentity_sort <- lapply(genes,function(x) paste(x,c('-sg1','-sg2','-sg3'),sep = '')) %>% unlist()
sgIdentity_sort <- c(
    expand.grid(
      level = c("_Use", "_top 20%"), 
      sg = c("-sg1", "-sg2", "-sg3")
    ) %>%
      apply(1, function(x) paste0("SOAT1", x["sg"], x["level"]))
)
sgIdentity_sort

In [ ]:
library(stringr)
data_plot <- lm_results_df %>% 
    mutate(
        p_log = -log10(empirical_p),
        is_sig = empirical_p < 0.01,
        term = term %>% str_remove('perturbation'),
        module = factor(module,levels = paste('MP',1:16,sep = ''))
    ) %>% 
    group_by(module) %>%
    mutate(
        term = case_when(
            term %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ term
        ),
        term = factor(term,levels = sgIdentity_sort),
        estimate_norm = estimate
    ) %>%
    group_by(module) %>% 
    mutate(
        estimate_scale = scale(estimate_norm)
    )
data_plot %>% head()

In [ ]:
data_plot %>% 
    filter(!is.na(term),module == 'MP12')

In [ ]:
colnames(data_plot)

In [ ]:
summary(data_plot$fdr_empirical_p)
summary(data_plot$estimate_norm)
summary(data_plot$estimate_scale)
data_plot %>% 
    filter(term == 'NT') %>% 
    pull(estimate_norm) %>% unique()

In [ ]:
data_plot <- data_plot %>% 
    filter(term != 'NT')

In [ ]:
data_plot_sub <- data_plot %>% 
    filter(module %in% c('MP1','MP7','MP12','MP16')) %>% 
    mutate(
        module = factor(module,levels = c('MP12','MP16','MP1','MP7') %>% rev()),
        facet_group = str_extract(term, "sg[1-3]")
    )

In [ ]:
grepl('SOAT1',data_plot_sub$term)
data_plot_sub$term

In [ ]:
getwd()

In [ ]:
data_plot_raw <- read.csv('./lm_results_df_Tatin_sgRNA_identity.csv',row.names = 1)%>% 
    mutate(
        p_log = -log10(empirical_p),
        is_sig = empirical_p < 0.01,
        term = term %>% str_remove('perturbation'),
        module = factor(module,levels = paste('MP',1:16,sep = ''))
    ) %>% 
    group_by(module) %>%
    mutate(
        term = case_when(
            term %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ term
        ),
        estimate_norm = estimate
    ) %>%
    group_by(module) %>% 
    mutate(
        estimate_scale = scale(estimate_norm)
    ) %>% 
    filter(module %in% c('MP12'),grepl('SOAT1',term)) %>% 
    mutate(
        module = factor(module,levels = c('MP12','MP16','MP1','MP7') %>% rev()),
        facet_group = str_extract(term, "sg[1-3]")
    )
data_plot_raw %>% head()

In [ ]:
data_plot_filter <- read.csv('./lm_results_df_Tatin_sgRNA_identity_alod4—Group.csv',row.names = 1)%>% 
    mutate(
        p_log = -log10(empirical_p),
        is_sig = empirical_p < 0.01,
        term = term %>% str_remove('perturbation'),
        module = factor(module,levels = paste('MP',1:16,sep = ''))
    ) %>% 
    group_by(module) %>%
    mutate(
        term = case_when(
            term %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ term
        ),
        estimate_norm = estimate
    ) %>%
    group_by(module) %>% 
    mutate(
        estimate_scale = scale(estimate_norm)
    ) %>% 
    filter(module %in% c('MP12'),grepl('_Use',term)) %>% 
    mutate(
        module = factor(module,levels = c('MP12','MP16','MP1','MP7') %>% rev()),
        facet_group = str_extract(term, "sg[1-3]")
    )
data_plot_filter %>% head()

In [ ]:
data_plot <- rbind(
    data_plot_raw %>% mutate(Group = 'Raw'),
    data_plot_filter %>% mutate(Group = 'Filter')
) %>% 
    mutate(
        sgIdentity = term %>% str_remove('_Use') %>% str_remove('SOAT1-'),
        sgIdentity = factor(sgIdentity,levels = paste('sg',1:3,sep = ''))
    )
data_plot %>% head()

In [ ]:
options(repr.plot.width = 8,repr.plot.height = 9)
size_sm = 16
size_lg = 20
range = range(data_plot$estimate_scale)
expand = abs(diff(range)) * 0.1
ggplot(data = data_plot,aes(x = estimate_scale, y = sgIdentity,color = Group)) + 
    geom_point(size = 28, aes(color = Group)) +
    geom_text(
        aes(label = format(estimate_scale,digits = 3), x = estimate_scale), 
        size = 6, hjust = 0.5,color = 'white'
    ) +
    scale_x_continuous("estimate_scale", limits = c(0.49, range[2] + expand*0.9),expand = c(0,0)) +
    theme_bw() + 
    theme(
        panel.border = element_blank(),
        panel.grid = element_blank(), 
        axis.text.x = element_text(size = size_sm), 
        axis.text.y = element_text(size = size_sm), 
        axis.title.x = element_text(size = size_lg), 
        axis.title.y = element_blank(), 
        strip.text = element_text(size = size_lg), strip.background = element_blank(), 
        axis.line = element_line(),
        legend.position = "none", 
        legend.text = element_text(size = size_sm), 
        legend.title = element_text(size = size_sm), 
        legend.key.size = unit(0.6,"lines"), 
        legend.background = element_blank(),
        plot.title = element_text(size = size_lg, hjust = 0.5)
    )

In [ ]:
segment_data <- data_plot %>%
    select(sgIdentity, Group, estimate_scale) %>%
    pivot_wider(names_from = Group, values_from = estimate_scale) %>%
    dplyr::mutate(
        mid_x = (Raw + Filter) / 2,  
        diff_label = round(Filter - Raw, 2)
    )
segment_data
data_plot

### fig3 k

In [ ]:
options(repr.plot.width = 12,repr.plot.height = 5)
ggplot() +
    geom_segment(
        data = segment_data,
        aes(x = Raw, xend = Filter, y = sgIdentity, yend = sgIdentity),
        inherit.aes = FALSE,
        color = "grey50",
        linewidth = 1.2
    ) +
    geom_point(
        data = data_plot,
        aes(x = estimate_scale, y = sgIdentity, color = paste(sgIdentity,Group,sep = '-')),
        size = 20
    ) +
    geom_text(
        data = segment_data,
        aes(x = mid_x, y = sgIdentity, label = diff_label),
        inherit.aes = FALSE,nudge_y = 0.2,
        color = "black",size = 5,fontface = "italic"
    ) +
    scale_x_continuous(name = "Effect Size", limits = c(0.49, max(data_plot$estimate_scale) * 1.1), expand = c(0, 0)) +
    scale_color_manual(values = c("#E64B35","#F4A89B","#8491B4","#C3CBE0","#91D1C2","#CDEBE4")) +
    theme_bw() + 
    theme(
        panel.border = element_blank(),
        panel.grid = element_blank(), 
        axis.text.x = element_text(size = size_sm), 
        axis.text.y = element_text(size = size_sm), 
        axis.title.x = element_text(size = size_lg), 
        axis.title.y = element_blank(), 
        strip.text = element_text(size = size_lg), strip.background = element_blank(), 
        axis.line = element_line(),
        legend.text = element_text(size = size_sm), 
        legend.title = element_text(size = size_sm), 
        legend.key.size = unit(0.6,"lines"), 
        legend.background = element_blank(),
        plot.title = element_text(size = size_lg, hjust = 0.5)
    )
ggsave('../result_figs/Effect_Size_SOAT1_filter.pdf',width = 12,height = 5)

In [ ]:
options(repr.plot.width = 26,repr.plot.height = 10)
DimPlot(seu, reduction = "umap_MP",group.by = c('batch','RNA_snn_res.0.7','function_type'),label = TRUE,label.size = 8) +
  theme(#aspect.ratio = 1,
        axis.text = element_blank(),
        axis.title = element_blank(),
        axis.ticks = element_blank())

In [ ]:
seu$batch %>% table()

In [ ]:
options(repr.plot.width = 10,repr.plot.height = 10)
DimPlot(seu, reduction = "umap_MP",group.by = c('batch'),label = FALSE,label.size = 8,pt.size = 2,alpha = 0.8) +
  guides(color = guide_legend(title = 'Group',override.aes = list(size = 5))) +
  scale_color_manual(values = c('#93B1D1','#94C3AB'),labels = c('Normal', 'Tatin')) +
  labs(title = '') +
  theme(
      aspect.ratio = 1,
      legend.text = element_text(size = 16),
      axis.text = element_blank(),
      axis.title = element_blank(),
      axis.ticks = element_blank())
ggsave('../result_figs/umap_batch.pdf',width = 10,height = 10)

In [ ]:
meta_data <- seu@meta.data
meta_data <- meta_data %>% 
    mutate(annotation = case_when(
        RNA_snn_res.0.7 %in% c(4,6) ~ 'Subset 1',
        RNA_snn_res.0.7 %in% c(1,7) ~ 'Subset 2',
        RNA_snn_res.0.7 %in% c(3,5) ~ 'Subset 3',
        RNA_snn_res.0.7 %in% c(0,2) ~ 'Subset 4'
    ))
meta_data$annotation %>% table()
meta_data %>% head()
seu@meta.data <- meta_data

In [ ]:
paletteer::paletteer_d("ggsci::default_jama")

### fig3 a

In [ ]:
options(repr.plot.width = 26,repr.plot.height = 10)
p1 <- DimPlot(seu, reduction = "umap_MP",group.by = c('annotation'),cols = c("#E59C83","#88A3D3","#7BA191","#9A8AB2"),label = FALSE,pt.size = 2.5) +
  theme(aspect.ratio = 1,
        axis.text = element_blank(),
        axis.title = element_blank(),
        axis.ticks = element_blank())
library(viridis)
p2 <- FeaturePlot(seu, features = c('MP12','MP1','MP16','MP7'), reduction = "umap_MP", ncol=2) &
  scale_color_viridis(option="B") &
  theme(aspect.ratio = 1, axis.text=element_blank(), axis.ticks=element_blank())
p1 | p2
ggsave('../result_figs/umap_and_4mp.pdf',width = 26,height = 10)

In [ ]:
meta_data$sgRNA_identity %>% unique()

In [ ]:
Idents(seu) <- seu$annotation
diff_gene <- FindAllMarkers(
    object = seu,
    logfc.threshold = 0.5,
    test.use = "wilcox",
    only.pos = TRUE
)

In [ ]:
saveRDS(diff_gene,'../result_figs/diffgene_4_subset.rds')

In [ ]:
diff_gene %>% 
    filter(abs(avg_log2FC)>1,p_val_adj<0.05) %>% 
    group_by(cluster) %>% 
    summarise(Counts = n())

In [ ]:
tmp <- diff_gene %>% 
    mutate(diff_pct = pct.1 - pct.2) %>% 
    filter(p_val_adj<0.05,pct.1>0.3) %>% 
    group_by(cluster) %>% 
    filter(!grepl('ENSG',gene)) %>% 
    arrange(desc(avg_log2FC)) %>% 
    slice(1:5) %>% 
    mutate(cluster = factor(cluster,levels = c('Subset 1','Subset 2','Subset 3','Subset 4'))) %>% 
    arrange(cluster,desc(avg_log2FC))
tmp

In [ ]:
options(repr.plot.width = 42, repr.plot.height = 30)
p <- suppressMessages(
  FeaturePlot(seu, features = tmp$gene, reduction = "umap_MP", ncol = 5,pt.size = 2.5) &
    scale_color_viridis(option = "B") &
    theme(aspect.ratio = 1, axis.text = element_blank(), axis.ticks = element_blank())
)
print(p)
ggsave('../result_figs/Featureplot_Subset_diffgene_top5.pdf',width = 42,height = 30)

### Fig3 c

In [ ]:
library(ggpubr)
library(paletteer)
options(repr.plot.width = 8.8,repr.plot.height = 7)
ggboxplot(
      data_plot,
      x="batch", y="value",
      color ="batch",
      width = 0.6,
      palette = c('#DE9980',"#799D8E","#859ECA","#9687AD"),
      add = "jitter",alpha = 0.9,
      xlab = F,  bxp.errorbar=T,
      bxp.errorbar.width=0.5,
      add.params = list(alpha = 0.2, size = 0.4),
      size=0.5, outlier.shape=NA,legend = "right") +
      guides(color = guide_legend(title = 'Group'))+
      stat_compare_means(
        label = "p.format",size = 6,hide.ns = TRUE,
        comparisons = (data_plot$batch %>% unique() %>% as.character()  %>% sort() %>% combn(m = 2,simplify = FALSE)),
        method = "wilcox.test",
      ) +
      facet_wrap(~Protein,ncol = 9,strip.position = 'left',scales = 'free') +
      theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )
ggsave(filename = '../result_figs/plot_protein_with_subsets_alod4-Myc.pdf',width = 8.8,height = 7)

In [ ]:
library(ggpubr)
library(paletteer)
options(repr.plot.width = 8.8,repr.plot.height = 7)
ggboxplot(
      data_plot,
      x="batch", y="value",
      color ="batch",
      width = 0.6,
      palette = c('#DE9980',"#799D8E","#859ECA","#9687AD"),
      add = "jitter",alpha = 0.9,
      xlab = F,  bxp.errorbar=T,
      bxp.errorbar.width=0.5, 
      add.params = list(alpha = 0.2, size = 0.4),
      size=0.5, outlier.shape=NA,legend = "right") +
      guides(color = guide_legend(title = 'Group'))+
      facet_wrap(~Protein,ncol = 9,strip.position = 'left',scales = 'free') +
      theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )
ggsave(filename = '../result_figs/plot_protein_with_subsets_alod4-Myc_without_p.pdf',width = 8.8,height = 7)

In [ ]:
DefaultAssay(seu) <- 'RNA'
seu

In [ ]:
mRNA_select <- c("CEBPA","MYC","HNF1A","HNF4A","GPX4","TP53","SOAT1","FASN","SREBF1","SREBF2","LDLR")

In [ ]:
tmp[grepl('SREBF1',tmp)]

In [ ]:
data_plot <- FetchData(object = seu,vars = c(mRNA_select,'annotation')) %>% 
    tidyr::pivot_longer(cols = mRNA_select,names_to = 'mRNA',values_to = 'value') %>% 
    rename(batch = annotation) %>% 
    mutate(
        batch = factor(batch,levels = c('Subset 4','Subset 3','Subset 2','Subset 1') %>% rev())
    )
data_plot %>% head()

In [ ]:
data_plot$batch %>% unique()
data_plot$mRNA %>% unique()

In [ ]:
getwd()

In [ ]:
library(ggpubr)
library(paletteer)
options(repr.plot.width = 42,repr.plot.height = 16)
ggboxplot(
      data_plot,
      x="batch", y="value",
      color ="batch",
      width = 0.6,
      palette = paletteer_d("ggsci::nrc_npg"),
      add = "jitter",
      xlab = F,  bxp.errorbar=T,
      bxp.errorbar.width=0.5, 
      add.params = list(alpha = 0.2, size = 0.4),
      size=0.5, outlier.shape=NA,legend = "right") +
      guides(color = guide_legend(title = 'Group'))+
      stat_compare_means(
        label = "p.format",size = 8,
        comparisons = (data_plot$batch %>% unique() %>% as.character()  %>% sort() %>% combn(m = 2,simplify = FALSE)),
        method = "wilcox.test",
      ) +
      facet_wrap(~mRNA,ncol = 6,strip.position = 'left',scales = 'free') +
      theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )

## Supplementary Fig. 12 a

In [ ]:
seu$sgRNA_identity %>% unique()

In [ ]:
meta_data <- seu@meta.data %>% 
    mutate(
        sgRNA_identity = case_when(
            sgRNA_identity %in% c('NT1','NT2','AAVS') ~ 'NT',
            TRUE ~ sgRNA_identity
        )
    )
chisq <- chisq.test(table(meta_data$annotation,meta_data$sgRNA_identity))
data_plot <- chisq$observed/chisq$expected %>% 
    as.matrix() %>% as.data.frame()
data_plot %>% head()

In [ ]:
data_ggplot %>% head()

In [ ]:
sgIdentity_use <- c(
    "SREBF2", "HMGCR", "SQLE", "INSIG1","LDLR", "NPC1L1", "NPC1","APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3","SOAT1"
) %>% lapply(.,function(x){paste(x,'-sg',1:3,sep = '')}) %>% unlist() %>% c('NT',.)
sgIdentity_use

In [ ]:
'SREBF2-sg1' %in% colnames(data_plot)

In [ ]:
data_ggplot$batch

In [ ]:
limit_min;limit_max

In [ ]:
data_ggplot <- data_plot %>%
  tibble::rownames_to_column('celltype') %>%
  reshape2::melt(id.var = 'celltype',variable.name = 'batch',value.name = 'Ro/e') %>% 
  mutate(
    batch = factor(batch,levels = sgIdentity_use),
    celltype = factor(celltype,levels = paste('Subset',1:4,sep = ' ') %>% rev()),
    label = case_when(
        (`Ro/e` > 1.5) ~ '+++',
        (`Ro/e` > 1.4) ~ '++',
        (`Ro/e` > 1.3) ~ '+',
        (`Ro/e` < 1.3) ~ ''
    )
  ) %>% 
  group_by(celltype) %>% 
  mutate(roe_scale = scale(`Ro/e`,center = FALSE)[,1]) 
data_ggplot$roe_scale <- data_ggplot$`Ro/e`
limit_min <- data_ggplot$roe_scale %>% min()
limit_max <- data_ggplot$roe_scale %>% max()
options(repr.plot.width = 32,repr.plot.height = 4)
ggplot(data = data_ggplot,
       mapping = aes(x = batch,y = celltype)) +
  geom_tile(aes(fill = roe_scale)) +
  geom_text(aes(label = label),size = 7.2) +
  guides(
    fill = guide_colorbar(title = 'Ro/e',title.vjust = 1)
  ) +
  scale_fill_gradientn(
      colors = c('#2B86A6', '#FDF9D9', '#E57168', '#81141a'),
      values = scales::rescale(c(0, 1, 1.5, 4)),
      limits = c(0, 4),
      oob = scales::squish,
      breaks = 0:4,
      labels = 0:4
    ) +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    axis.title = element_blank(),
    axis.line = element_blank(),
    axis.text = element_text(size = 20),
    axis.text.x = element_text(angle = 30,hjust = 1),
    axis.text.y = element_text(hjust = 1),
    legend.text = element_text(size = 16),
    legend.position = 'right',
    legend.justification = c(0,1),
    legend.title = element_text(size = 20),
    strip.text = element_blank(),
    strip.background = element_blank()
  )
ggsave('../result_figs/Roe_sg1-3.pdf',width = 32,height = 4)

In [ ]:
data_ggplot %>% filter(batch == 'NT')

In [ ]:
DefaultAssay(seu) <- 'Protein'
seu <- NormalizeData(object = seu,normalization.method = "CLR", margin =2)
seu

In [ ]:
colnames(seu@meta.data )

In [ ]:
data_plot <-  FetchData(seu, vars = c('annotation','MP1','MP7','MP12','MP16',rownames(seu)),layer = 'data') %>% 
    tibble::rownames_to_column('Sample') %>% 
    tidyr::pivot_longer(cols = rownames(seu),names_to = 'Protein',values_to = 'Expression') %>% 
    tidyr::pivot_longer(cols = c('MP1','MP7','MP12','MP16'),names_to = 'Module',values_to = 'UCell_module') %>% 
    dplyr::filter(
        (annotation == 'Subset 1' & Module == 'MP12') |
        (annotation == 'Subset 2' & Module == 'MP1') |
        (annotation == 'Subset 3' & Module == 'MP16') |
        (annotation == 'Subset 4' & Module == 'MP7')
    )
data_plot %>% head()

In [ ]:
library(ggplot2)
library(ggpmisc) 

In [ ]:
r2_table <- data_plot %>%
  group_by(Protein, annotation) %>%
  summarise(
    r2 = summary(lm(Expression ~ UCell_module))$r.squared,
    pval = summary(lm(Expression ~ UCell_module))$coefficients[2, 4],
    n = n()
  ) %>% 
    arrange(desc(r2))
r2_table

In [ ]:
options(repr.plot.width = 42,repr.plot.height = 7*30)
ggplot(data = data_plot,aes(x = UCell_module,y = Expression)) +
    geom_point() +
    geom_smooth(method = "lm",formula = y ~ x, se = FALSE, color = "blue") +
    stat_poly_eq(
        aes(label = paste(after_stat(eq.label), ..rr.label.., ..p.value.label.., sep = "~~~")),
        formula = y ~ x,
        parse = TRUE,size = 8
    ) +
    facet_wrap(Protein ~ annotation,ncol = 4,strip.position = 'left',scales = 'free') +
    theme_classic() +
    theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
    )
ggsave('../result_figs/protein_Subset_cor.pdf',width = 42,height = 210,limitsize = FALSE)

In [ ]:
data_plot_sub <- data_plot %>% 
    filter(annotation == 'Subset 4' & Protein == 'ALOD4-pAbO')
data_plot_sub %>% head()

In [ ]:
data_plot_heatmap <- data_plot %>% 
    group_by(annotation,Protein)%>%
    summarise(
        spearman_r = cor(Expression, UCell_module, method = "spearman", use = "complete.obs"),
        # n = n(),
        .groups = "drop"
    ) %>% 
    arrange(annotation,desc(abs(spearman_r))) %>% 
    group_by(annotation) %>% 
    pivot_wider(names_from = Protein, values_from = spearman_r) %>% 
    tibble::column_to_rownames('annotation') %>% 
    as.matrix()
data_plot_heatmap %>% head()

In [ ]:
library(ComplexHeatmap)
library(circlize)
library(dendextend)

In [ ]:
col_fun <- colorRamp2(
  c(0, data_plot_heatmap %>% max() %>% {./2}, data_plot_heatmap %>% max() %>% {.*1.2}),
  c("white", "skyblue", "darkblue"))

In [ ]:
options(repr.plot.width = 24,repr.plot.height = 4)
ht <- Heatmap(
  data_plot_heatmap,
  name = "Jaccard",
  col = col_fun,
  cluster_rows = TRUE,
  cluster_columns = TRUE,
  show_row_dend = TRUE,
  show_column_dend = TRUE,
  row_names_gp = gpar(fontsize = 6),
  column_names_gp = gpar(fontsize = 16)
)
ht_built <- draw(ht, merge_legend = TRUE)
row_dend <- row_dend(ht_built)
col_dend <- column_dend(ht_built)
row_hc <- as.hclust(row_dend)
col_hc <- as.hclust(col_dend)
k_row <- 4
k_col <- 8

row_dend <- as.dendrogram(row_hc)
col_dend <- as.dendrogram(col_hc)

row_dend_colored <- color_branches(row_dend, k = k_row)
col_dend_colored <- color_branches(col_dend, k = k_col)

row_groups <- cutree(row_hc, k = k_row)
col_groups <- cutree(col_hc, k = k_col)

In [ ]:
col_fun <- colorRamp2(
  c(data_plot_heatmap %>% min() %>% {.*1.2},0, data_plot_heatmap %>% max() %>% {.*1.2}),
  c("#26456E","white",  "#9C0824"))

In [ ]:
data_plot_heatmap

In [ ]:
options(repr.plot.width = 24,repr.plot.height = 7)
ht_all <- Heatmap(
    data_plot_heatmap,
    name = "Spearman",
    col = col_fun,na_col = "white",
    cell_fun = function(j,i,x,y,width,height,fill){
    grid.text(
        label = sprintf("%.2f",data_plot_heatmap[i,j]),
        x=x,y = y,gp = gpar(fontsize = 12,col = "black")
    )  
    },
    cluster_rows = FALSE,
    cluster_columns = col_dend_colored,
    column_split = k_col,
    show_row_dend = TRUE,
    show_column_dend = TRUE,
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    show_heatmap_legend = TRUE,
    heatmap_legend_param = list(
        title = 'Spearman',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',
        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)
pdf('../result_figs/spearman_protein_Subset.pdf',width = 24,height = 7)
draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))
dev.off()
draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))

In [ ]:
data_plot_heatmap %>% colnames()

In [ ]:
Protein_select <- c(
        'ALOD4-pAbO','pAKT-pAbO','pP65-pAbO','p-RPS6-pAbO','p-STAT3-pAbO','pERK-pAbO','pCREB-pAbO','pGSK3b-pAbO',
        'PDL1-pAbO','c-MYc-pAbO','SOX2-pAbO','Vimentin-pAbO','GPX4-pAbO'
     )

In [ ]:
data_plot_heatmap_sub <- data_plot_heatmap %>% 
    as.data.frame() %>% 
    t() %>% as.data.frame()
data_plot_heatmap_sub <- scale(data_plot_heatmap_sub, center = FALSE, scale = apply(abs(data_plot_heatmap_sub), 2, max)) %>% 
    t() %>% as.data.frame() %>% select(all_of(Protein_select)) %>% t()

data_plot_heatmap_sub

In [ ]:
col_fun <- colorRamp2(
  c(data_plot_heatmap_sub %>% min() %>% {.*1},0, data_plot_heatmap_sub %>% max() %>% {.*1}),
  c("#88A3D3","white",  "#BA6B6E"))

## Fig.3 d

In [ ]:
options(repr.plot.width = 9,repr.plot.height = 14)
ht_all <- Heatmap(
    data_plot_heatmap_sub,
    name = "Spearman",
    col = col_fun,na_col = "white",
    cell_fun = function(j,i,x,y,width,height,fill){
        grid.text(
            label = sprintf("%.2f",data_plot_heatmap_sub[i,j]),
            x=x,y = y,gp = gpar(fontsize = 12,col = "black")
        )  
    },
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    column_names_side = "top",
    row_names_side = "left",
    show_row_dend = TRUE,
    show_column_dend = TRUE,
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    show_heatmap_legend = TRUE,
    heatmap_legend_param = list(
        title = 'Norm. correlation',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',
        legend_height = unit(70, units = "mm"),
        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)
pdf('../result_figs/spearman_select_protein_Subset.pdf',width = 9,height = 14)
draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))
dev.off()
draw(ht_all,padding = unit(c(20, 20, 20, 40), "mm"))

In [ ]:
rownames(seu)

In [ ]:
data_module <- read.csv('../result_figs/data_module_Genes_all.csv',row.names = 1) %>% 
    group_by(Genes) %>% 
    slice(1) %>% 
    ungroup() %>% 
    mutate(Module = factor(Module,levels = paste('MP',1:16,sep = ''))) %>% 
    arrange(Module) %>% 
    mutate(Genes = factor(Genes,levels = Genes))
data_module %>% dim()
data_module %>% head()

In [ ]:
data_module <- read.csv('../result_figs/data_module_Genes_all.csv',row.names = 1) %>%
    group_by(Module) %>% 
    arrange(desc(Module_score),.by_group = TRUE) %>% 
    mutate(Module = factor(Module,levels = paste('MP',1:16,sep = ''))) %>% 
    group_by(Genes) %>% 
    mutate(n = n()) %>% 
    filter(n<2 | Module_score == max(Module_score)) %>% 
    group_by(Module) %>% 
    arrange(desc(Module_score)) %>%
    slice(if (cur_group()$Module == "MP16") 1:20 else 1:10)
data_module %>% dim()
data_module$Module %>% table()
data_module$Genes %>% duplicated() %>% table()
data_module %>% head()

In [ ]:
DefaultAssay(seu) <- 'RNA'
Idents(seu) <- seu$annotation
seu

In [ ]:
data_module %>% head()

In [ ]:
data_plot <- DotPlot(object = seu,features = data_module$Genes %>% unique(),assay = 'RNA',group.by = 'annotation')$data %>% 
    rename(Genes = features.plot) %>% 
    mutate(
        Genes = Genes %>% as.character()
    ) %>% 
    left_join(data_module %>% mutate(Genes = Genes %>% as.character()), by = 'Genes') %>% 
    mutate(
        Genes = factor(Genes,levels = data_module$Genes),
        Module
    )
data_plot %>% dim()
data_plot %>% head()

In [ ]:
data_plot %>% head()

In [ ]:
options(repr.plot.width = 7,repr.plot.height = 18)
ggplot(
    data = data_plot %>% 
        filter(Module %in% c('MP1','MP7','MP12','MP16')) %>% 
    mutate(Module = factor(Module,levels = c('MP12','MP16','MP1','MP7'))),
    mapping = aes(x = id,y = Genes,color = avg.exp.scaled,size = pct.exp)
) +
    geom_point() +
    guides(
        color = guide_colorbar(title = 'Avg Expression',title.vjust = 1,title.position = "left",
      title.theme = element_text(angle = 90, vjust = 0.5))
      ) +
    scale_color_gradientn(
      colors = c('#2B86A6', '#FDF9D9', '#E57168', '#81141a'),
      limits = c(-1.5, 1.5),
      oob = scales::squish,
      breaks = c(-1.5, 0,1.5),
      labels = c(-1.5, 0,1.5)
    ) +
    facet_wrap( ~ Module,scales = 'free_y',ncol = 1,switch = 'strip.position') +
    theme_classic() +
    theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(0.8, "cm"),
        axis.text.x = element_text(size = 24,angle = 45,hjust = 1,vjust = 1),
        axis.title = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
    )
ggsave('../result_figs/dotplot_mpgenes.pdf',width = 7,height = 18)

In [ ]:
data_module %>% filter(Module == 'MP12')

In [ ]:
diffgene_all <- FindAllMarkers(object = seu,assay = 'RNA',slot = 'data',min.pct = 0,only.pos = TRUE)
diffgene_all %>% head()

In [ ]:
data_diff <- diffgene_all %>% 
    group_by(cluster) %>% 
    filter(p_val<0.01) %>% 
    arrange(avg_log2FC,.by_group = TRUE) %>% 
    rename('Genes' = 'gene','Module' = 'cluster') %>% 
    mutate(
        Module = factor(Module,levels = paste('Subset ',1:4,sep = ''))
    ) %>% 
    arrange(Module,avg_log2FC)
data_diff %>% dim()
data_diff %>% head()

In [ ]:
gene_use <- data_plot %>% 
    dplyr::filter(Module %in% c('MP1','MP7','MP12','MP16')) %>% 
    pull(Genes)
gene_use %>% length()

In [ ]:
table(gene_use %in% data_diff$Genes)

In [ ]:
data_diff %>% filter(Genes %in% gene_use) %>% pull(avg_log2FC) %>% min()

In [ ]:
data_diff <- data_diff %>% 
    filter(avg_log2FC>0.1)

In [ ]:
my36colors <-c(
  "#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b",
  "#e377c2","#7f7f7f","#bcbd22","#17becf","#aec7e8","#ffbb78",
  "#98df8a","#ff9896","#c5b0d5","#c49c94","#f7b6d2","#c7c7c7",
  "#dbdb8d","#9edae5","#7698b3","#d6616b","#a55194","#ce6dbd",
  "#756bb1","#8c6d31","#b5cf6b","#7b4173","#cedb9c","#6b6ecf",
  "#9c9ede","#bd9e39","#d9d9d9","#ad494a","#8ca252","#e7ba52"
) %>% sample(36,replace = FALSE)

In [ ]:
data_diff$Genes %>% unique() %>% length()

In [ ]:
data_module_use <- data_module %>% 
    filter(Module %in% c('MP1','MP7','MP12','MP16')) %>% 
    mutate(Module = factor(Module,levels = c('MP12','MP16','MP1','MP7')))
data_module_use %>% dim()
data_module_use %>% head()

In [ ]:
data_module_use %>% 
    filter(!(Genes %in% data_diff$Genes))
'AFF3' %in% data_diff$Genes
'AFF3' %in% rownames(seurat_obj)

In [ ]:
data_diff_use <- data_diff %>%
  group_by(Genes) %>%
  mutate(n = n()) %>% 
  filter(n < 2 | avg_log2FC == max(avg_log2FC)) %>%  
  select(-n) %>% 
  ungroup() %>% 
  arrange(Module,desc(avg_log2FC))
data_diff %>%  dim()
data_diff_use %>%  dim()
data_diff_use %>%  head()

In [ ]:
data_diff_use %>% 
    group_by(Genes) %>% 
    mutate(n = n()) %>% 
    filter(n>=2) %>% 
    arrange(Genes)

In [ ]:
data_plot <- DotPlot(object = seu,features = data_diff_use$Genes %>% unique(),assay = 'RNA',group.by = 'annotation')$data %>% 
    rename(Genes = features.plot) %>% 
    mutate(
        Genes = Genes %>% as.character()
    ) %>% 
    left_join(data_module %>% mutate(Genes = Genes %>% as.character()), by = 'Genes') %>% 
    mutate(
        Genes = factor(Genes,levels = data_diff_use$Genes %>% unique())
    )
data_plot %>% dim()
data_plot %>% head()

In [ ]:
data_plot <- data_plot %>% 
    select(c('avg.exp.scaled','Genes','id')) %>% 
    pivot_wider(names_from = 'id',values_from = 'avg.exp.scaled') %>% 
    mutate(Genes = factor(Genes,levels = data_diff_use$Genes)) %>% 
    arrange(Genes) %>% 
    column_to_rownames('Genes') %>% 
    select(paste('Subset',1:4))
data_plot %>% dim()
data_plot %>% head()

In [ ]:
data_module_use <- data_module %>% 
    filter(Module %in% c('MP1','MP7','MP12','MP16')) %>% 
    mutate(Module = factor(Module,levels = c('MP12','MP16','MP1','MP7'))) %>% 
    filter((Genes %in% rownames(data_plot))) %>% 
    arrange(Module,desc(Module_score))
data_module_use %>% dim()
data_module_use %>% head()

In [ ]:
data_plot %>% dim()
data_plot %>% head()

In [ ]:
data_diff_use %>% head()

In [ ]:
library(ComplexHeatmap)
library(grid)
group_df <- data_diff_use %>%
  mutate(group = Module) %>%
  arrange(group) %>%
  distinct(Genes, .keep_all = TRUE) %>%
  select(Genes, group)
dim(group_df)
rownames(group_df) <- group_df$Genes

color_use <- my36colors[1:length(unique(group_df$group))]
names(color_use) <- levels(group_df$group)
group_vec <- factor(group_df$group, levels = paste("Subset", 1:4))

group_annotation <- rowAnnotation(
  group = anno_block(
    gp = gpar(fill = color_use[levels(group_vec)]),
    labels = levels(group_vec),
    labels_gp = gpar(fontsize = 12),
    labels_rot = 0  
  ),
  show_annotation_name = FALSE,
  width = unit(10, "mm")
)

In [ ]:
data_diff_use$Module %>% unique()
data_module_use %>% head()
data_module_use %>% dim()
data_module_use$Module %>% unique()

In [ ]:
options(repr.plot.width = 12,repr.plot.height = 48)
ht <- Heatmap(
    matrix = data_plot,
    col = colorRamp2(c(data_plot %>% min() %>% {./2},0, data_plot %>% max() %>% {.*1.2}),c("white", "skyblue", "darkblue")),
    row_split = group_vec,
    row_title_gp = gpar(fontsize = 0),
    column_title_gp = gpar(fontsize = 0),
    row_gap = unit(3, "mm"),
    column_gap = unit(2, "mm"),
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    show_column_names = TRUE,
    show_row_names = TRUE,
    heatmap_legend_param = list(
        title = 'Relative Expression',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',
        legend_height = unit(80, units = "mm"),
        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)

In [ ]:
data_module_use$Genes %>% .[!grepl('ENSG',.)] %>% length()
data_module_use$Genes %>% length()

In [ ]:
data_module_use

In [ ]:
modules <- unique(data_module_use$Module)
module_colors <- setNames(
    c('#DE9980',"#799D8E","#859ECA","#9687AD"),
    modules
)
genes_for_mark <- data_module_use$Genes[!grepl('ENSG', data_module_use$Genes)] %>% as.character()
gene_module_map <- data_module_use %>% 
    filter(Genes %in% genes_for_mark) %>%
    select(Genes, Module) %>% 
    mutate(Genes = factor(Genes,levels = rownames(data_plot)[which(rownames(data_plot) %in% genes_for_mark)])) %>% 
    arrange(Genes)
labels_colors <- module_colors[gene_module_map$Module]

row_anno <- rowAnnotation(
  link = anno_mark(
    at = which(rownames(data_plot) %in% genes_for_mark), 
    labels = rownames(data_plot)[which(rownames(data_plot) %in% genes_for_mark)],
    labels_gp = gpar(fontsize = 20, col = labels_colors)
  )
)

In [ ]:
module_colors <- setNames(
  colorRampPalette(RColorBrewer::brewer.pal(length(modules), "Set2"))(length(group_vec %>% levels())),
  group_vec %>% levels()
)
group_annotation <- rowAnnotation(
  group = anno_block(
    gp = gpar(fill = module_colors[levels(group_vec)]),
    labels = levels(group_vec),
    labels_gp = gpar(fontsize = 28,col = 'white'),
    labels_rot = 90  
  ),
  show_annotation_name = FALSE,
  width = unit(10, "mm")
)
group_legend <- Legend(
  labels = levels(group_vec),
  legend_gp = gpar(fill = module_colors[levels(group_vec)]),
  title = "Group",   
  title_gp = gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
  labels_gp = gpar(fontsize = 24, col = "black"),
  title_position = "leftcenter-rot",
  grid_width = unit(11, "mm"),    
  grid_height = unit(12, "mm")
)

In [ ]:
top_annotation <- HeatmapAnnotation(
  group = anno_block(
    gp = gpar(fill = module_colors[levels(group_vec)]),
    labels = levels(group_vec),
    labels_gp = gpar(fontsize = 28, col = 'white'),
    labels_rot = 0  
  ),
  show_annotation_name = FALSE,
  height = unit(10, "mm")
)

In [ ]:
module_colors

## fig3b  left

In [ ]:
options(repr.plot.width = 12,repr.plot.height = 32)
ht <- Heatmap(
    matrix = data_plot %>% as.matrix(),
    col = colorRamp2(c(data_plot %>% min() %>% {./2},0, data_plot %>% max() %>% {.*1.2}),c("white", "#C0D2E3", "#5586B4")),
    top_annotation = top_annotation,
    left_annotation = group_annotation,
    column_split = paste('Subset',1:4),
    row_split = group_vec,
    row_title_gp = gpar(fontsize = 0),
    column_title_gp = gpar(fontsize = 0),
    row_gap = unit(3, "mm"),
    column_gap = unit(2, "mm"),
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    show_column_names = TRUE,
    show_row_names = FALSE,
    show_heatmap_legend = FALSE,
    heatmap_legend_param = list(
        title = 'Relative Expression',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',
        legend_height = unit(80, units = "mm"),
        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)
heatmap_legend <- Legend(
  title = 'Relative Expression',
  at = c(-1,0,1,2),
  col_fun = colorRamp2(
    c(data_plot %>% min() %>% {./2}, 0, data_plot %>% max() %>% {.*1.2}),
    c("white", "#C0D2E3", "#5586B4")
  ),
  legend_height = unit(80, "mm"),grid_width = unit(12,"mm"),
  direction = "vertical",
  title_gp = gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
  labels_gp = gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
  title_position = "leftcenter-rot"
)
pdf('../result_figs/heatmap_top10Genes.pdf',width = 12,height = 32)
draw(
  ht, 
  annotation_legend_list = list(group_legend, heatmap_legend),
  merge_legend = TRUE
)
dev.off()
draw(
  ht,
  annotation_legend_list = list(group_legend, heatmap_legend),
  merge_legend = TRUE
)

In [ ]:
options(repr.plot.width = 12,repr.plot.height = 32)
ht <- Heatmap(
    matrix = data_plot %>% as.matrix(),
    col = colorRamp2(c(data_plot %>% min() %>% {./2},0, data_plot %>% max() %>% {.*1.2}),c("white", "skyblue", "darkblue")),
    left_annotation = group_annotation,
    row_split = group_vec,
    row_title_gp = gpar(fontsize = 0),
    column_title_gp = gpar(fontsize = 0),
    row_gap = unit(3, "mm"),
    column_gap = unit(2, "mm"),
    row_names_gp = gpar(fontsize = 26),
    column_names_gp = gpar(fontsize = 26),
    cluster_rows = TRUE,
    cluster_columns = FALSE,
    show_column_names = TRUE,
    show_row_names = TRUE,
    heatmap_legend_param = list(
        title = 'Relative Expression',
        title_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5),
        title_position = 'leftcenter-rot',
        legend_direction = 'vertical',
        legend_height = unit(80, units = "mm"),
        labels_gp = grid::gpar(fontsize = 24, fontface = "bold", hjust = 0.5, vjust = 0.5)
    )
)
draw(ht + row_anno)

# endline